In [8]:
import time
import os
import cv2
import glob
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from ultralytics import YOLO
import rasterio
from rasterio.transform import from_origin
from rasterio.windows import Window
from PIL import Image
from datetime import timedelta
from tqdm.notebook import tqdm
import gc

# ======================== Sliding Window Reader ========================
def slide_geotiff(geotiff_path, window_size, stride):
    tiles = []
    with rasterio.open(geotiff_path) as src:
        height, width = src.height, src.width
        transform, crs, dtype, count = src.transform, src.crs, src.dtypes[0], src.count

        for y in range(0, height - window_size + 1, stride):
            for x in range(0, width - window_size + 1, stride):
                window = Window(x, y, window_size, window_size)
                slice_img = src.read(window=window)

                new_transform = from_origin(
                    transform.c + x * transform.a,
                    transform.f + y * transform.e,
                    transform.a,
                    -transform.e
                )

                tiles.append({
                    "img_array": slice_img,
                    "transform": new_transform,
                    "crs": crs,
                    "dtype": dtype,
                    "count": count,
                    "position": (y, x)
                })

    return tiles


# ======================== Overlay and Extraction ========================
def overlay_segmentation_on_array(results, transform, img_shape, tif_image):
    object_data = []
    height, width, _ = img_shape

    if results.masks is None:
        return object_data

    for mask, box, cls, conf in zip(results.masks.xy, results.boxes.xyxy, results.boxes.cls, results.boxes.conf):
        cls = int(cls.item())
        x1, y1, x2, y2 = map(int, box.tolist())

        mask_layer = np.zeros((height, width), dtype=np.uint8)
        if mask is not None and len(mask) > 0:
            cv2.fillPoly(mask_layer, [np.array(mask, dtype=np.int32)], 255)

        pixel_count = np.count_nonzero(mask_layer[y1:y2, x1:x2])
        area = pixel_count * (transform.a * abs(transform.e))

        lon, lat = transform * ((x1 + x2) / 2, (y1 + y2) / 2)

        object_data.append({
            "latitude": lat,
            "longitude": lon,
            "area": area,
            "pixel_count": pixel_count,
            "class": cls,
            "confidence": float(conf.item()),
            "tile_name": tif_image
        })

    return object_data

# ======================== Utilities ========================
def find_tif_images(folder_path):
    return glob.glob(os.path.join(folder_path, '**', '*.tif'), recursive=True)


# ======================== Maximum Likelihood Suppression ========================
from ultralytics.engine.results import Results
from ultralytics.utils import ops

def maximum_likelihood_suppression(results):
    if results.masks is None or results.boxes is None:
        return results

    # Extract values
    conf = results.boxes.conf
    cls = results.boxes.cls

    selected_indices = []
    seen_classes = set()
    sorted_indices = torch.argsort(conf, descending=True)

    for idx in sorted_indices:
        class_id = int(cls[idx].item())
        if class_id not in seen_classes:
            selected_indices.append(idx)
            seen_classes.add(class_id)

    selected_indices = torch.tensor(selected_indices, dtype=torch.long)

    # Assign filtered data (only `.data` is writeable)
    results.boxes.data = results.boxes.data[selected_indices]
    if results.masks is not None and results.masks.data is not None:
        results.masks.data = results.masks.data[selected_indices]
        # Skip assigning to results.masks.xy – it's read-only and not essential

    return results


import torchvision.ops as ops

def global_mls_by_mask_iou(object_infos, iou_threshold=0.5):
    if not object_infos:
        return []

    # Group by class
    df = pd.DataFrame(object_infos)
    retained_infos = []

    for class_id in df['class'].unique():
        group = df[df['class'] == class_id]

        if len(group) == 1:
            retained_infos.append(group.iloc[0].to_dict())
            continue

        # Convert masks to bounding boxes
        boxes = []
        for _, row in group.iterrows():
            x = row['longitude']
            y = row['latitude']
            w = h = 0.99  # very small box, approx. because we don't have mask pixel positions
            boxes.append([x, y, x + w, y + h])

        boxes_tensor = torch.tensor(boxes, dtype=torch.float32)
        scores_tensor = torch.tensor(group['confidence'].values, dtype=torch.float32)

        keep_indices = ops.nms(boxes_tensor, scores_tensor, iou_threshold)

        retained_infos.extend(group.iloc[keep_indices.numpy()].to_dict(orient='records'))

    return retained_infos



In [9]:
# ======================== Load YOLO Model ========================
model = YOLO("C:/Users/mza0288/Documents/CODE/runs/segment/train29/weights/best.pt")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print(f"Using device: {device}")

# ======================== Parameters ========================
window_size = 3000
stride = 1500
batch_size = 8
# folder_path = 'E:/Data/al_30cm_2021_transfer_2_of_2/'
folder_path = './geotiff'

# ======================== Main Inference Loop ========================
tif_images = find_tif_images(folder_path)
accumulate_object_infos = []
total_images = len(tif_images)

overall_bar = tqdm(
    enumerate(tif_images, 1), 
    total=total_images, 
    desc="Overall Progress",
    unit="image",
    bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]'
)

for idx, tif_image in overall_bar:
    image_start_time = time.time()
    slices = slide_geotiff(tif_image, window_size=window_size, stride=stride)
    slice_batches = [slices[i:i + batch_size] for i in range(0, len(slices), batch_size)]

    batch_bar = tqdm(
        slice_batches,
        desc=f"Processing {os.path.basename(tif_image)}",
        leave=False,
        unit="batch",
        bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]'
    )

    for batch in batch_bar:
        pil_imgs_batch = []
        for slice_data in batch:
            img_array = slice_data["img_array"]
            rgb_img = np.moveaxis(img_array[:3], 0, -1)
            pil_imgs_batch.append(Image.fromarray(rgb_img))

        results_batch = model(pil_imgs_batch, device=device, batch=batch_size, verbose=False)

        for slice_data, results in zip(batch, results_batch):
            # print(f"Before MLS: {len(results.boxes)} boxes")
            results = maximum_likelihood_suppression(results)
            # print(f"After MLS: {len(results.boxes)} boxes")
            object_info = overlay_segmentation_on_array(
                results,
                transform=slice_data["transform"],
                img_shape=rgb_img.shape,
                tif_image=os.path.basename(tif_image)
            )
            accumulate_object_infos.extend(object_info)
            # After processing all slices of current image:
            accumulate_object_infos = global_mls_by_mask_iou(accumulate_object_infos)

    # Save intermediate results
    # intermediate_csv = f"data/sub_2/intermediate_object_info_{idx}.csv"
    # pd.DataFrame(accumulate_object_infos).to_csv(intermediate_csv, index=False)

    # Cleanup
    del slices, slice_batches, batch_bar, pil_imgs_batch, results_batch, img_array, rgb_img
    torch.cuda.empty_cache()
    gc.collect()

    image_time_taken = time.time() - image_start_time
    overall_bar.set_postfix({
        'Last_image_time': f"{timedelta(seconds=int(image_time_taken))}",
        'Images_left': total_images - idx,
    })

overall_bar.close()

final_csv_path = "data/temp.csv"
pd.DataFrame(accumulate_object_infos).to_csv(final_csv_path, index=False)
print(f"\n✅ Final results saved to {final_csv_path}")


Using device: cuda


Overall Progress:   0%|          | 0/1 [00:00<?, ?image/s]

Processing m_3008501_ne_16_030_20211103.tif:   0%|          | 0/25 [00:00<?, ?batch/s]


✅ Final results saved to data/temp.csv
